In [2]:
import numpy as np
import pandas as pd

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [7]:
df=pd.read_csv('covid_toy.csv')

In [8]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [9]:
##Step-1:Train and split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [10]:
X_train

,age,gender,fever,cough,city
2,42,Male,101.0,Mild,Delhi
20,12,Male,98.0,Strong,Bangalore
26,19,Female,100.0,Mild,Kolkata
65,69,Female,102.0,Mild,Bangalore
87,47,Male,101.0,Strong,Bangalore
...,...,...,...,...,...
74,34,Female,104.0,Strong,Delhi
83,17,Female,104.0,Mild,Kolkata
72,83,Female,101.0,Mild,Kolkata
47,18,Female,104.0,Mild,Bangalore


In [12]:
##without column transfer
##Adding simple imputer to fever column
si=SimpleImputer()
X_train_fever=si.fit_transform(X_train[['fever']])
X_test_fever=si.fit_transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [15]:
#Ordinalencoding for cough column
oe=OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough=oe.fit_transform(X_train[['cough']])
X_test_cough=oe.fit_transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [18]:
#OneHotEncoding-gender,city
ohe=OneHotEncoder(drop='first',sparse_output=False)##for multicollinearity
X_train_gender_city=ohe.fit_transform(X_train[['gender','city']])
X_test_gender_city=ohe.fit_transform(X_test[['gender','city']])
X_train_gender_city.shape

(80, 4)

In [20]:
#Extracting Age
X_train_age=X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age=X_test.drop(columns=['gender','fever','cough','city']).values
X_train_age.shape

(80, 1)

In [21]:

##merge all columns
X_train_transformed=np.concatenate((X_train_age,X_train_fever,X_train_cough,X_train_gender_city),axis=1)
X_test_transformed=np.concatenate((X_test_age,X_test_fever,X_test_cough,X_test_gender_city),axis=1)
X_train_transformed.shape

(80, 7)

In [22]:
##Using column transfer
from sklearn.compose import ColumnTransformer 

In [27]:
transformer=ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
    
],remainder='passthrough')

In [29]:
transformer.fit_transform(X_train).shape

(80, 7)

In [30]:
transformer.transform(X_test).shape

(20, 7)